# Track A: The Physics of Encoded Magic States

**Plan C — Parallel Tracks**

This track is pure quantum mechanics. It covers the theory behind magic states, the [[4,2,2]] stabilizer code, and the witness formula. No optimization, no scoring — just the physics.

> **Dashboard:** Open `00_dashboard.ipynb` alongside this notebook for interactive exploration.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from math import pi, sqrt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp, state_fidelity, Operator
from qiskit.visualization import plot_bloch_multivector
from qiskit_aer import AerSimulator

from autoresearch_quantum.codes.four_two_two import (
    build_preparation_circuit, build_encoder, apply_magic_seed,
    encoded_magic_statevector, STABILIZERS, MEASUREMENT_OPERATORS, DATA_QUBITS,
)
from autoresearch_quantum.experiments.encoded_magic_state import build_circuit_bundle
from autoresearch_quantum.models import ExperimentSpec
from autoresearch_quantum.execution.analysis import logical_magic_witness

print("All imports successful.")

In [ ]:
from autoresearch_quantum.teaching import LearningTracker
from autoresearch_quantum.teaching.assess import quiz, predict_choice, reflect, order, checkpoint_summary
tracker = LearningTracker("plan_c_track_a")
print("Learning tracker active.")

---
## 1. Why Magic States Matter

Quantum error correction can protect information, but it has a fundamental limitation:

> **The Eastin-Knill Theorem:** No quantum error-correcting code can implement a universal gate set transversally (i.e., by applying independent gates to each physical qubit).

Clifford gates ($H$, $S$, CNOT) *can* be done transversally on many codes. But Cliffords alone are **classically simulable** (Gottesman-Knill theorem). To get universal quantum computation, you need at least one non-Clifford gate.

The standard choice is the **T-gate** ($\pi/8$ rotation):

$$T = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\pi/4} \end{pmatrix}$$

Instead of applying $T$ directly (which would break error correction), we prepare a special resource state called a **magic state** and consume it via **gate teleportation**.

In [ ]:
quiz(tracker, "q1_eastin_knill",
    question="The Eastin-Knill theorem limits fault-tolerant QC. What does it say?",
    options=[
        "No quantum code can detect all errors",
        "No quantum code has a universal set of transversal gates \u2014 you need a non-transversal resource like magic states",
        "Quantum error correction always requires more physical qubits than logical qubits",
    ],
    correct=1, section="1. Why magic states", bloom="remember",
    explanation="Eastin-Knill: you cannot implement a universal gate set transversally in any code. The T-gate is the most common non-transversal resource, supplied via magic states.")

---
## 2. The T-State on the Bloch Sphere

The magic state for the T-gate is:

$$|T\rangle = H \cdot T|0\rangle = \frac{|0\rangle + e^{i\pi/4}|1\rangle}{\sqrt{2}}$$

In [ ]:
qc = QuantumCircuit(1, name="|T>")
qc.h(0)
qc.p(pi/4, 0)

t_state = Statevector.from_instruction(qc)
print("T-state amplitudes:")
print(f"  |0>: {t_state[0]:.4f}")
print(f"  |1>: {t_state[1]:.4f}")
print(f"  |1> phase: {np.angle(t_state[1]) * 180 / pi:.1f} degrees = pi/4")

alpha, beta = t_state[0], t_state[1]
exp_x = 2 * np.real(np.conj(alpha) * beta)
exp_y = 2 * np.imag(np.conj(alpha) * beta)
exp_z = float(np.abs(alpha)**2 - np.abs(beta)**2)
print(f"\nBloch coordinates:")
print(f"  <X> = {exp_x:.4f}  (expected: 1/sqrt(2) = {1/sqrt(2):.4f})")
print(f"  <Y> = {exp_y:.4f}  (expected: 1/sqrt(2) = {1/sqrt(2):.4f})")
print(f"  <Z> = {exp_z:.4f}  (on the equator)")

plot_bloch_multivector(t_state)

In [ ]:
quiz(tracker, "q2_tstate_phase",
    question="The T-state phase on |1\u27E9 is:",
    options=["\u03C0/2", "\u03C0/4", "\u03C0/8"],
    correct=1, section="2. T-state", bloom="remember",
    explanation="\u03C0/4 = 45\u00b0. Despite the gate being called T (\u03C0/8 rotation on the Bloch sphere), the state phase is \u03C0/4.")

> **Key Insight:** The T-state sits at $\theta = \pi/2$ from the Z-axis and $\phi = \pi/4$ azimuthally. Its defining feature: $\langle X \rangle = \langle Y \rangle = 1/\sqrt{2}$. No stabilizer state has this property.

---
## 3. Three Equivalent Preparations

The codebase offers three gate sequences to prepare $|T\rangle$. All produce the same state up to a global phase.

In [ ]:
styles = ["h_p", "ry_rz", "u_magic"]
states = []

for style in styles:
    qc = QuantumCircuit(1)
    apply_magic_seed(qc, 0, style)
    sv = Statevector.from_instruction(qc)
    states.append(sv)
    print(f"{style:8s}: amplitudes = [{sv[0]:.4f}, {sv[1]:.4f}]")

print("\nPairwise fidelities (1.0 = identical up to global phase):")
for i in range(len(styles)):
    for j in range(i+1, len(styles)):
        fid = state_fidelity(states[i], states[j])
        print(f"  {styles[i]} vs {styles[j]}: F = {fid:.10f}")

In [ ]:
quiz(tracker, "q3_global_phase",
    question="Three gate sequences produce states with different amplitudes but fidelity 1.0. Why?",
    options=[
        "Floating-point errors",
        "Global phase has no physical consequence",
        "They actually produce different states",
    ],
    correct=1, section="3. Preparations", bloom="understand",
    explanation="A global phase multiplies ALL amplitudes. No measurement can distinguish the states.")
checkpoint_summary(tracker, "3. Preparations")

---
## 4. The [[4,2,2]] Code

The [[4,2,2]] code encodes **2 logical qubits** into **4 physical qubits** with distance **2**.

The **stabilizer group**:

$$\mathcal{S} = \langle X_0 X_1 X_2 X_3,\; Z_0 Z_1 Z_2 Z_3 \rangle$$

The codespace is the simultaneous $+1$ eigenspace of both generators. Dimension: $2^4 / 2^2 = 4 = 2^2$ (room for 2 logical qubits).

In [ ]:
for name, stab in STABILIZERS.items():
    print(f"{name}: {stab.to_list()}")
    stab_sq = stab @ stab
    is_identity = np.allclose(stab_sq.to_matrix(), np.eye(16))
    print(f"  Squares to identity: {is_identity}")

comm = STABILIZERS["z_stabilizer"] @ STABILIZERS["x_stabilizer"] - STABILIZERS["x_stabilizer"] @ STABILIZERS["z_stabilizer"]
print(f"\n[ZZZZ, XXXX] = {np.max(np.abs(comm.to_matrix())):.1e} (should be 0 = they commute)")

In [ ]:
quiz(tracker, "q4_stabilizer_square",
    question="Each stabilizer squares to the identity (S\u00b2 = I). What does this imply about its eigenvalues?",
    options=[
        "Eigenvalues can be anything",
        "Eigenvalues are exactly +1 or \u22121",
        "Eigenvalues are 0 or 1",
    ],
    correct=1, section="4. [[4,2,2]] code", bloom="understand",
    explanation="If S\u00b2 = I, then S has eigenvalues \u00b11. The codespace has eigenvalue +1; error states have \u22121.")

---
## 5. Logical Operators

| Logical qubit | $X_L$ | $Z_L$ |
|---|---|---|
| Qubit 0 (magic) | $X_0 X_2$ | $Z_0 Z_2$ |
| Qubit 1 (spectator) | $X_1 X_3$ | $Z_1 Z_2$ |

For the magic witness we measure: $X_L$, $Y_L = Y_0 Z_1 X_2$, and $Z_{\text{spectator}} = Z_1 Z_2$.

In [ ]:
print("Measurement operators for the magic witness:")
for name, op_dict in MEASUREMENT_OPERATORS.items():
    pauli_str = ["I"] * 4
    for qubit, basis in op_dict.items():
        pauli_str[qubit] = basis
    label = "".join(reversed(pauli_str))
    print(f"  {name:15s} = {label}  (qubits {dict(op_dict)})")

In [ ]:
quiz(tracker, "q5_logical_ops",
    question="Why does the logical Y operator (Y\u2080Z\u2081X\u2082) involve 3 qubits instead of just 1?",
    options=[
        "It's a bug in the code",
        "In a quantum code, logical operators act on the encoded information which is spread across multiple physical qubits",
        "Y is always a 3-qubit operator",
    ],
    correct=1, section="5. Logical operators", bloom="understand",
    explanation="The logical information is distributed across all physical qubits. Logical operators must act on this distributed encoding.")
checkpoint_summary(tracker, "5. Logical operators")

---
## 6. The Encoding Circuit

In [ ]:
for style in ["cx_chain", "cz_compiled"]:
    enc = build_encoder(style)
    print(f"\n{style}:")
    print(enc.draw("text"))

In [ ]:
# Full preparation: seed + encoder
prep = build_preparation_circuit("h_p", "cx_chain")
prep.draw("mpl", style="iqp")

---
## 7. Verifying the Encoded State

The encoded T-state must satisfy: $\langle XXXX \rangle = \langle ZZZZ \rangle = +1$ (in codespace) and $\langle X_L \rangle = \langle Y_L \rangle = 1/\sqrt{2}$, $\langle Z_{\text{spectator}} \rangle = +1$.

In [ ]:
state = encoded_magic_statevector()

print("Stabilizer expectations:")
for name, stab in STABILIZERS.items():
    val = state.expectation_value(stab).real
    print(f"  <{name}> = {val:+.6f}  (should be +1)")

print("\nLogical operator expectations:")
for name, op_dict in MEASUREMENT_OPERATORS.items():
    pauli_str = ["I"] * 4
    for qubit, basis in op_dict.items():
        pauli_str[qubit] = basis
    op = SparsePauliOp.from_list([("".join(reversed(pauli_str)), 1.0)])
    val = state.expectation_value(op).real
    print(f"  <{name}> = {val:+.6f}")

print(f"\nExpected: <X_L> = <Y_L> = 1/sqrt(2) = {1/sqrt(2):+.6f}")

In [ ]:
predict_choice(tracker, "q6_z_error",
    question="A single Z error on qubit 0: which stabilizer detects it?",
    options=[
        "ZZZZ (Z commutes with Z, so it detects Z errors)",
        "XXXX (Z anti-commutes with X, flipping the XXXX eigenvalue)",
        "Neither \u2014 Z errors are invisible",
    ],
    correct=1, section="8. Error detection", bloom="apply",
    explanation="Z anti-commutes with X. A Z error on any qubit flips the XXXX eigenvalue from +1 to \u22121.")

---
## 8. Error Detection

The [[4,2,2]] code detects any single-qubit error. Let us apply errors and see which stabilizer flags them.

In [ ]:
header = f"{'Error':12s} {'<ZZZZ>':>8s} {'<XXXX>':>8s} {'Detected by':>15s}"
print(header)
print("=" * len(header))

for qubit in range(4):
    for error_name, error_gate in [("X", "x"), ("Z", "z"), ("Y", "y")]:
        test_circuit = build_preparation_circuit("h_p", "cx_chain")
        getattr(test_circuit, error_gate)(qubit)
        errored_state = Statevector.from_instruction(test_circuit)

        z_exp = errored_state.expectation_value(STABILIZERS["z_stabilizer"]).real
        x_exp = errored_state.expectation_value(STABILIZERS["x_stabilizer"]).real

        detected = []
        if abs(z_exp - 1.0) > 0.01: detected.append("ZZZZ")
        if abs(x_exp - 1.0) > 0.01: detected.append("XXXX")

        print(f"{error_name} on q{qubit}:   {z_exp:+.1f}     {x_exp:+.1f}     {', '.join(detected) or '(none)'}")

In [ ]:
order(tracker, "q7_error_types",
    instruction="Sort error types by how many stabilizers they trigger (fewest first):",
    items=["X", "Z", "Y"],
    correct_order=["X", "Z", "Y"],
    section="8. Error detection", bloom="analyze",
    explanation="X\u21921 (ZZZZ). Z\u21921 (XXXX). Y\u21922 (both). X and Z are tied.")
checkpoint_summary(tracker, "8. Error detection")

> **Key Insight:** X errors flip ZZZZ to $-1$. Z errors flip XXXX to $-1$. Y = iXZ flips both. Every single-qubit error is detected by at least one stabilizer.

---
## 9. The Magic Witness Formula

$$W = \frac{1 + \frac{\langle X_L \rangle + \langle Y_L \rangle}{\sqrt{2}}}{2} \times \frac{1 + \langle Z_{\text{spectator}} \rangle}{2}$$

- Magic factor: checks non-Clifford character of logical qubit 0
- Spectator factor: checks logical qubit 1 is undisturbed
- $W = 1.0$ for perfect encoded T-state

In [ ]:
lx = 1/sqrt(2)
ly = 1/sqrt(2)
sz = 1.0

magic_factor = (1 + (lx + ly)/sqrt(2)) / 2
spectator_factor = (1 + sz) / 2
W = magic_factor * spectator_factor

print(f"<X_L> = {lx:.4f},  <Y_L> = {ly:.4f},  <Z_spectator> = {sz:.4f}")
print(f"Magic factor:     (1 + ({lx:.4f}+{ly:.4f})/sqrt(2)) / 2 = {magic_factor:.4f}")
print(f"Spectator factor: (1 + {sz:.4f}) / 2 = {spectator_factor:.4f}")
print(f"Witness W = {W:.4f}")
print(f"Library:  W = {logical_magic_witness(lx, ly, sz):.4f}")

In [ ]:
quiz(tracker, "q8_ideal_witness",
    question="For a perfect T-state, the magic witness W equals:",
    options=["0.0", "0.5", "1/\u221A2", "1.0"],
    correct=3, section="9. Witness formula", bloom="apply",
    explanation="Ideal values give magic_factor = 1 and spectator_factor = 1. Product = 1.0.")

---
## 10. How the Witness Degrades

In [ ]:
lx_values = np.linspace(-1, 1, 200)
w_vals = [logical_magic_witness(lx, lx, 1.0) for lx in lx_values]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lx_values, w_vals, "b-", linewidth=2)
ax.axvline(x=1/np.sqrt(2), color="r", linestyle="--", label="T-state: 1/√2")
ax.set_xlabel("<X_L> = <Y_L>")
ax.set_ylabel("Witness W")
ax.set_title("Magic Witness vs Logical Operator Expectations")
ax.legend()
ax.set_xlim(-1, 1)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
reflect(tracker, "q9_witness_sensitivity",
    question="The witness curve drops sharply away from the peak. Why is this useful?",
    section="10. Witness degradation", bloom="evaluate",
    model_answer="A sharp peak means the witness is sensitive to small deviations from the ideal T-state. This sensitivity is what makes it a good diagnostic: even moderate noise produces a noticeable drop.")
checkpoint_summary(tracker, "10. Witness degradation")

> **Observe:** The witness peaks sharply at $1/\sqrt{2}$. Any deviation from the ideal T-state expectation values reduces $W$.

---
## Summary

| Concept | Key fact |
|---|---|
| **Magic states** | Non-Clifford resource for universal QC |
| **T-state** | $\langle X \rangle = \langle Y \rangle = 1/\sqrt{2}$ on the Bloch equator |
| **[[4,2,2]] code** | 4 qubits, 2 logical, distance 2, stabilizers XXXX and ZZZZ |
| **Error detection** | X caught by ZZZZ, Z caught by XXXX, Y caught by both |
| **Magic witness** | $W=1$ certifies genuine encoded T-state |

> **Next:** Track B covers noise and engineering. Track C covers automated search.

### Free Response: Physics Synthesis

Reflect on the key physics concepts from this track.

---
## Final Assessment

In [ ]:
tracker.dashboard()
path = tracker.save()
print(f"\nProgress saved to: {path}")

---
## Navigation — Plan C

**→ Next: [Track B — Engineering](track_b_engineering.ipynb)**

*← [Dashboard](00_dashboard.ipynb) · [Start Here](../00_START_HERE.ipynb)*